In [1]:
# Cell 1: Install dependencies
!pip install -U transformers
!pip install fastapi uvicorn nest-asyncio
!pip install -U bitsandbytes>=0.46.1
# Download cloudflared for tunneling (no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

print('✅ All dependencies installed')


cloudflared version 2026.6.0 (built 2026-06-09-09:33 UTC)
✅ All dependencies installed


In [ ]:
# Cell 2: Load Qwen3-14B model with 4-bit quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

import nest_asyncio
nest_asyncio.apply()

import uvicorn
import subprocess
import re
import time
import threading
import asyncio
import select
import uuid
import logging
import traceback
import gc

from fastapi import FastAPI, Request
from pydantic import BaseModel


MODEL_PATH = '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1'

print(f'Loading model from {MODEL_PATH}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Use 4-bit quantization to shrink model from 29GB -> 9GB
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=quantization_config,
    device_map='auto',
)
model.eval()

print(f'\n✅ Model loaded successfully!')
print(f'Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB')


Loading model from /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]


✅ Model loaded successfully!
Model memory footprint: 5.96 GB


In [ ]:
# Cell 3: Start FastAPI server + Cloudflare tunnel with verbose logging



# =============================================================================
# LOGGING CONFIGURATION
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger("QWEN_API")

# =============================================================================
# FASTAPI APP
# =============================================================================

app = FastAPI(title="Kaggle Qwen3-14B API")


class GenerationRequest(BaseModel):
    prompt: str
    system: str = ""
    max_tokens: int = 4096
    temperature: float = 0.7


# =============================================================================
# GPU UTILITIES
# =============================================================================

def gpu_status(prefix=""):
    try:
        if not torch.cuda.is_available():
            logger.info("CUDA not available")
            return

        logger.info(f"{prefix} GPU STATUS")

        for i in range(torch.cuda.device_count()):
            allocated = torch.cuda.memory_allocated(i) / 1024**3
            reserved = torch.cuda.memory_reserved(i) / 1024**3

            logger.info(
                f"GPU {i} | "
                f"Allocated={allocated:.2f}GB | "
                f"Reserved={reserved:.2f}GB"
            )

    except Exception as e:
        logger.error(f"GPU status failed: {e}")


def cleanup_gpu():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# =============================================================================
# GENERATION
# =============================================================================

from transformers.generation.streamers import BaseStreamer
from tqdm.auto import tqdm

class ProgressBarStreamer(BaseStreamer):
    def __init__(self, max_tokens, request_id=""):
        self.max_tokens = max_tokens
        self.request_id = request_id
        self.pbar = None
        self.is_prompt = True

    def put(self, value):
        if self.is_prompt:
            self.is_prompt = False
            self.pbar = tqdm(total=self.max_tokens, desc=f"[{self.request_id}] Gen", unit="tok", leave=False)
            return
        if self.pbar is not None:
            num_tokens = value.numel() if hasattr(value, 'numel') else 1
            self.pbar.update(num_tokens)

    def end(self):
        if self.pbar is not None:
            self.pbar.close()


def generate_response(
    prompt,
    system="",
    max_tokens=4096,
    temperature=0.7,
    request_id="unknown"
):
    start_time = time.time()

    logger.info("=" * 80)
    logger.info(f"[{request_id}] GENERATION STARTED")

    gpu_status(f"[{request_id}] BEFORE GENERATION")

    messages = []

    if system:
        messages.append(
            {
                "role": "system",
                "content": system
            }
        )

    messages.append(
        {
            "role": "user",
            "content": prompt
        }
    )

    logger.info(
        f"[{request_id}] Applying chat template..."
    )

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    logger.info(
        f"[{request_id}] Tokenizing input..."
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    input_tokens = inputs["input_ids"].shape[1]

    logger.info(
        f"[{request_id}] Input Tokens: {input_tokens}"
    )

    logger.info(
        f"[{request_id}] Starting model.generate()"
    )

    streamer = ProgressBarStreamer(max_tokens, request_id)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=max(temperature, 0.01),
            do_sample=temperature > 0.01,
            top_p=0.9,
            repetition_penalty=1.1,
            streamer=streamer,
        )

    logger.info(
        f"[{request_id}] Generation finished."
    )

    generated_ids = outputs[0][input_tokens:]

    generated_tokens = len(generated_ids)

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    elapsed = time.time() - start_time

    logger.info(
        f"[{request_id}] Generated Tokens: {generated_tokens}"
    )

    logger.info(
        f"[{request_id}] Total Time: {elapsed:.2f}s"
    )

    logger.info(
        f"[{request_id}] RESPONSE PREVIEW:"
    )

    logger.info(
        response[:1000]
    )

    gpu_status(f"[{request_id}] AFTER GENERATION")

    logger.info("=" * 80)

    return response


# =============================================================================
# API ROUTES
# =============================================================================

@app.post("/generate")
async def api_generate(
    request_data: GenerationRequest,
    request: Request
):
    request_id = str(uuid.uuid4())[:8]

    try:
        logger.info("\n")
        logger.info("#" * 100)
        logger.info(f"[{request_id}] NEW REQUEST RECEIVED")
        logger.info("#" * 100)

        client_ip = request.client.host

        logger.info(
            f"[{request_id}] Client IP: {client_ip}"
        )

        logger.info(
            f"[{request_id}] Max Tokens: {request_data.max_tokens}"
        )

        logger.info(
            f"[{request_id}] Temperature: {request_data.temperature}"
        )

        logger.info(
            f"[{request_id}] Prompt Length: {len(request_data.prompt)} chars"
        )

        logger.info(
            f"[{request_id}] USER PROMPT:"
        )

        logger.info(
            request_data.prompt[:3000]
        )

        if request_data.system:
            logger.info(
                f"[{request_id}] SYSTEM PROMPT:"
            )

            logger.info(
                request_data.system[:3000]
            )

        response = generate_response(
            prompt=request_data.prompt,
            system=request_data.system,
            max_tokens=request_data.max_tokens,
            temperature=request_data.temperature,
            request_id=request_id,
        )

        logger.info(
            f"[{request_id}] REQUEST COMPLETED SUCCESSFULLY"
        )

        return {
            "request_id": request_id,
            "response": response,
        }

    except Exception as e:
        logger.error(
            f"[{request_id}] ERROR OCCURRED"
        )

        logger.error(traceback.format_exc())

        return {
            "request_id": request_id,
            "error": str(e),
        }


@app.get("/")
def health_check():
    return {
        "status": "running",
        "model": "Qwen3-14B",
        "gpus": torch.cuda.device_count(),
    }


# =============================================================================
# START UVICORN
# =============================================================================

def start_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info",
    )

    server = uvicorn.Server(config)

    loop.run_until_complete(server.serve())


server_thread = threading.Thread(
    target=start_server,
    daemon=True,
)

server_thread.start()

time.sleep(3)

print("\n✅ FastAPI server started on port 8000")


# =============================================================================
# START CLOUDFLARE TUNNEL
# =============================================================================

print("\n🌍 Starting Cloudflare Tunnel...")

tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8000",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(10)

logs = ""

while select.select(
    [tunnel_process.stderr],
    [],
    [],
    0.1
)[0]:
    logs += tunnel_process.stderr.read1(4096).decode()

url_match = re.search(
    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
    logs
)

if not url_match:
    print("⏳ Waiting for tunnel...")
    time.sleep(5)

    while select.select(
        [tunnel_process.stderr],
        [],
        [],
        0.1
    )[0]:
        logs += tunnel_process.stderr.read1(4096).decode()

    url_match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        logs
    )

if url_match:
    public_url = url_match.group()

    print("\n" + "=" * 80)
    print("🚀 PUBLIC API READY")
    print("=" * 80)

    print(f"Base URL : {public_url}")
    print(f"Endpoint : {public_url}/generate")

    print("\nENV VARIABLE:")
    print(f"KAGGLE_API_URL={public_url}")

    print("=" * 80)

else:
    print("\n❌ FAILED TO DETECT TUNNEL URL")
    print(logs)


# =============================================================================
# SELF TEST
# =============================================================================

import requests

time.sleep(2)

print("\n🧪 Running self-test...")

try:
    response = requests.post(
        "http://localhost:8000/generate",
        json={
            "prompt": "Say hello in one sentence.",
            "max_tokens": 50,
            "temperature": 0.7,
        },
        timeout=300,
    )

    result = response.json()

    print("\n✅ SELF TEST SUCCESS")

    if "response" in result:
        print(
            "\nResponse:\n",
            result["response"][:500]
        )

except Exception as e:
    print(f"\n❌ Self-test failed: {e}")


# =============================================================================
# KEEP NOTEBOOK ALIVE
# =============================================================================

print("\n" + "=" * 80)
print("✅ SERVER RUNNING")
print("📌 Keep notebook open")
print("📌 Cloudflare URL active while Kaggle session lives")
print("📌 Every request will be logged below")
print("=" * 80)

try:
    while True:
        time.sleep(60)

except KeyboardInterrupt:
    print("\nStopping server...")
    tunnel_process.terminate()

INFO:     Started server process [276]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ FastAPI server started on port 8000

🚀 API is live at: https://toilet-radical-classification-ash.trycloudflare.com
👉 POST to: https://toilet-radical-classification-ash.trycloudflare.com/generate

Update your .env with:
KAGGLE_API_URL=https://toilet-radical-classification-ash.trycloudflare.com
INFO:     127.0.0.1:41554 - "POST /generate HTTP/1.1" 200 OK

🧪 Self-test result: Hello! How can I assist you today?

✅ Server is running! Keep this notebook open.
The server will stay alive as long as the Kaggle session is active (~12 hours max).


INFO:     2401:4900:1c6a:2f7e:4884:abe0:e9e:46b:0 - "GET / HTTP/1.1" 200 OK
INFO:     2401:4900:1c6a:2f7e:4884:abe0:e9e:46b:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     122.173.28.38:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     122.173.28.38:0 - "POST /extract HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /extract HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /extract HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /extract HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /extract HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /extract HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /extract HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /extract HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /extract HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /extract HTTP/1.1" 404 Not Found
INFO:     122.173.28.38:0 - "POST /ext